# 📰🤖 Notícias AO VIVO + SEU FinBERT → direção (SmartTrader)

Junta tudo: busca notícias reais do momento, carrega o **seu modelo treinado**
(do Drive) e gera a **direção final calibrada** (comprar/vender) por ativo.
Autossuficiente — não precisa clonar repositório. Rode no Colab (GPU ajuda).

> ⚠️ Honestidade: é um **filtro de viés macro**, não bola de cristal. O FinBERT
> dá o sentimento; as regras macro dão a direção; juntos calibram a confiança.
> Valide em conta DEMO antes de qualquer dinheiro real.


## Passo 1 — Instalar (sem mexer no torch)


In [ ]:
!pip install -q -U transformers feedparser requests
import torch; print('GPU:', torch.cuda.is_available())


## Passo 2 — Montar o Drive e carregar o SEU FinBERT
Usa o modelo salvo em `MyDrive/finbert_ft`. Se não achar, cai no FinBERT padrão.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
from transformers import AutoTokenizer, AutoModelForSequenceClassification

M = '/content/drive/MyDrive/finbert_ft'
if not os.path.exists(M):
    print('⚠️ Modelo do Drive nao encontrado — usando FinBERT padrao (ProsusAI).')
    M = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(M)
model = AutoModelForSequenceClassification.from_pretrained(M); model.eval()
print('Modelo carregado de:', M)

def sentimento_conf(texto):
    x = tok(texto, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad(): p = torch.softmax(model(**x).logits, 1)[0]
    return float(p.max())   # forca do sentimento (0..1)


## Passo 3 — Lógica: notícias → temas → direção calibrada
Versão fiel ao `smarttrader/` (frequência de documentos + conflito + teto 0.85).


In [ ]:
import feedparser

DEFAULT_RSS_FEEDS = [
    'https://finance.yahoo.com/news/rssindex',
    'https://www.cnbc.com/id/100003114/device/rss/rss.html',
    'https://www.cnbc.com/id/10000664/device/rss/rss.html',
    'https://www.investing.com/rss/news_285.rss',
    'https://www.investing.com/rss/news_1.rss',
]
def fetch_rss(urls):
    out = []
    for u in urls:
        try:
            for e in feedparser.parse(u).entries:
                out.append((e.get('title','') + ' ' + e.get('summary','')).strip())
        except Exception as ex:
            print('feed falhou:', u, ex)
    return out

SYMBOL_QUERY = {
  'USDCAD':['oil','crude','opec','canada','canadian dollar','boc'],
  'USOIL':['oil','crude','opec','brent','wti','energy'],
  'XAUUSD':['gold','inflation','war','safe haven','rates'],
  'EURUSD':['euro','ecb','eurozone'],
  'SPX':['stocks','s&p','wall street','recession','earnings'],
  'BTCUSD':['bitcoin','crypto','etf'],
}
THEME_KEYWORDS = {
  'oil':['oil','crude','opec','brent','wti','petroleum','barrel'],
  'rates_hawkish':['rate hike','hike rates','tightening','hawkish','inflation surge','higher for longer'],
  'rates_dovish':['rate cut','cut rates','easing','dovish','inflation cooling','disinflation'],
  'risk_off':['war','crisis','conflict','recession','crash','selloff','sell-off','turmoil','geopolitical','sanctions'],
  'risk_on':['rally','optimism','soft landing','risk appetite','record high','bull market','rebound'],
}
UP=['surge','spike','soar','jump','rise','rally','climb','gain','escalate','higher','up']
DOWN=['plunge','plummet','drop','fall','slump','glut','oversupply','crash','slide','tumble','ease','cool','lower','down']
SYMBOL_EXPOSURE = {
  'EURUSD':{'rates_hawkish':-0.8,'rates_dovish':0.8,'risk_off':-0.5,'risk_on':0.4,'oil':-0.15},
  'USDCAD':{'oil':-0.7,'risk_off':0.6,'risk_on':-0.4,'rates_hawkish':0.5,'rates_dovish':-0.5},
  'XAUUSD':{'risk_off':0.8,'risk_on':-0.4,'rates_hawkish':-0.6,'rates_dovish':0.6,'oil':0.2},
  'USOIL':{'oil':1.0,'risk_off':0.3,'risk_on':0.3},
  'SPX':{'risk_off':-0.8,'risk_on':0.7,'rates_hawkish':-0.4,'rates_dovish':0.4},
  'BTCUSD':{'risk_off':-0.5,'risk_on':0.6},
}
K=4.0; MIN_CONF=0.15; CONF_SCALE=0.8; MAX_BASE=0.85

def detect_themes(texts):
    docs=[t.lower() for t in texts if t and t.strip()]
    if not docs: return []
    blob=' '.join(docs)
    up=sum(blob.count(w) for w in UP); down=sum(blob.count(w) for w in DOWN)
    out=[]
    for theme,kws in THEME_KEYWORDS.items():
        dh=sum(1 for d in docs if any(k in d for k in kws))
        if not dh: continue
        strength=dh/(dh+K)
        s = 1 if up>down else (-1 if down>up else (0 if theme=='oil' else 1))
        out.append((theme,s,strength))
    return out

def bias_for_symbol(symbol, themes):
    exp=SYMBOL_EXPOSURE.get(symbol)
    if not exp or not themes: return 0,0.0,'sem tema/exposicao'
    net=0.0; pos=0.0; neg=0.0; contribs=[]
    for theme,s,strength in themes:
        c=exp.get(theme)
        if c is None: continue
        contrib=c*s*strength
        if contrib==0: continue
        net+=contrib; contribs.append((theme,contrib))
        if contrib>0: pos+=contrib
        else: neg+=-contrib
    if not contribs: return 0,0.0,'sem exposicao relevante'
    total=pos+neg; conf=min(abs(net)/CONF_SCALE, MAX_BASE)
    opp = pos>0 and neg>0 and (min(pos,neg)/total)>=0.35
    if opp and conf<MIN_CONF: return 0,round(conf,2),f'forcas conflitantes se anulam (alta={pos:.2f} x baixa={neg:.2f})'
    if conf<MIN_CONF: return 0,round(conf,2),f'forca macro fraca ({net:+.2f})'
    bias=1 if net>0 else -1
    det='; '.join(f'{t}({c:+.2f})' for t,c in sorted(contribs,key=lambda x:-abs(x[1])))
    return bias,round(conf,2),f'net={net:+.2f}; {det}'


## Passo 4 — Rodar AO VIVO (notícia real + seu modelo)


In [ ]:
itens = fetch_rss(DEFAULT_RSS_FEEDS)
print(f'Manchetes reais puxadas agora: {len(itens)}')

def vies_ao_vivo(symbol):
    kws = SYMBOL_QUERY.get(symbol, [])
    rel = [t for t in itens if any(k in t.lower() for k in kws)] if kws else itens
    if not rel: return 'NEUTRO', 0.0, 'sem noticia relevante', 0
    themes = detect_themes(rel)
    bias, conf, rat = bias_for_symbol(symbol, themes)
    # REFINA a confianca com o SEU FinBERT (sentimento medio das manchetes)
    if bias != 0 and rel:
        amostra = rel[:20]
        sent = sum(sentimento_conf(t) for t in amostra) / len(amostra)
        conf = round(max(0.0, min(1.0, conf*(0.5 + 0.5*sent))), 2)
    lado = {1:'COMPRA', -1:'VENDA', 0:'NEUTRO'}[bias]
    return lado, conf, rat, len(rel)

print('\n=== DIRECAO FINAL (notica ao vivo + seu FinBERT) ===')
for s in ['USOIL','XAUUSD','USDCAD','EURUSD','SPX','BTCUSD']:
    lado, conf, rat, n = vies_ao_vivo(s)
    print(f'  {s:7s} {lado:7s} conf={conf:.2f}  ({n} manchetes)  | {rat[:70]}')


## Pronto! 🎉
Isto é a sua IA completa: lê o mundo ao vivo, usa o **seu modelo** e decide direção.
No bot (no PC), o mesmo é feito por `news_ai.bias_from_live_news(symbol, scorer=fb)`.
Próximo passo do projeto: ligar isso ao **MT5 demo** (Modo A) e validar por semanas.
